# Assignment: Build an AI Student Helpdesk Assistant



## Business Scenario
Northstar Learning Hub has a handbook containing its student policies. The academic team wants a basic AI assistant that can answer learners' questions **using the handbook**, instead of guessing.

You will build a small pipeline that:
1. Reads the provided PDF and divides it into text chunks.
2. Tests **zero-shot and few-shot prompts** to categorize student questions.
3. Calls the **DeepSeek API** from Python.
4. Retrieves a relevant handbook chunk using **keyword overlap**.
5. Uses a **LangChain chain** to generate a grounded answer.
6. Tests whether answers are supported by the PDF, including for missing information.


## Learning Outcomes


By completing this assignment, you should be able to:
- Apply system/user messages and zero-shot/few-shot prompting.
- Send API requests using `requests` with configurable `temperature` and `max_tokens`.
- Extract and chunk PDF text with Python and `pypdf`.
- Retrieve a relevant chunk with simple word-overlap scoring.
- Connect `PromptTemplate`, `RunnableLambda`, and `StrOutputParser`.
- Evaluate grounded answers and identify unsupported responses.



## Provided resources




- [Northstar_Student_Handbook.pdf](https://drive.google.com/file/d/13GKO3S8Z0ek6nOHqZeizAbqu3X9IDcsi/view?usp=drive_link)

- Example test questions near the end


## Submission



Upload your completed `.ipynb` with executed outputs on GitHub and share the link of repository.

**Note on API provider:** The assignment specifies the DeepSeek API. My DeepSeek requests returned `HTTP 402 Payment Required` because the account needs a paid balance. So I used NVIDIA's hosted Nemotron model (`nvidia/nemotron-3-ultra-550b-a55b`), which is available through a free trial endpoint with an OpenAI-compatible chat API.

Everything else follows the assignment: `requests` for the API call, the same `temperature` and `max_tokens` settings, and the same function structure. The API function is named `call_nvidia` instead of `call_deepseek`, and the API key is stored in Colab Secrets as `NVIDIA_API_KEY`.

## Part A — Prepare Your Colab Environment
### Step 0. Install libraries and load your API key

Run the setup cells below. The installer and key-loading code are provided because the assessed work begins with the functions in Step 1.

1. In Colab, open the **Secrets** sidebar.
2. Create a secret named `DEEPSEEK_API_KEY` and allow this notebook to use it.
3. Never paste your secret into a code cell or output.
4. Use a DeepSeek model available to your account. The default shown below is `deepseek-chat`; update it only if your account requires a different supported model.

**Purpose of Code:** This setup cell installs the required libraries, imports the tools used throughout the assignment, and loads your DeepSeek API key securely. Later functions depend on these imports and settings to run.


In [119]:
%pip -q install requests pypdf langchain-core

import os
import re
import requests
from typing import Any
from pypdf import PdfReader
from google.colab import files, userdata
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise ValueError("Save NVIDIA_API_KEY in Colab Secrets and enable access.")

NVIDIA_API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
NVIDIA_MODEL = "nvidia/nemotron-3-ultra-550b-a55b"
HEADERS = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Content-Type": "application/json",
}
print("Setup complete. API key loaded securely.")

Setup complete. API key loaded securely.


In [120]:
print(call_nvidia([{"role": "user", "content": "Say hello in one word."}]))

Hello


### Step 1. Upload the student handbook PDF

Download the **provided** handbook PDF and upload it in the next cell. Run this once each time the Colab runtime is reset. We use the file's original name in all later steps.

**Purpose of Code:** This cell makes the handbook PDF available inside the Colab runtime and saves its file path. The PDF-reading function in the next step will use this path to open the correct document.


In [116]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the supplied student handbook PDF.")

PDF_PATH = next(iter(uploaded))
if not PDF_PATH.lower().endswith(".pdf"):
    raise ValueError("The uploaded file must be a PDF.")
print("Uploaded:", PDF_PATH)

Saving Northstar_Student_Handbook (1).pdf to Northstar_Student_Handbook (1) (3).pdf
Uploaded: Northstar_Student_Handbook (1) (3).pdf


### Step 2. Extract text from the PDF

**Your task:** Complete `read_handbook_pdf(pdf_path)`.

- Open the file with `PdfReader`.
- Extract the text from every page.
- Ignore any empty page text.
- Join the extracted text into **one string**.
- Return the combined string.

**Expected input:** Path to one text-based PDF.  
**Expected output:** A non-empty string containing handbook text.

**Hint:** Recall `page.extract_text()` from the LangChain & RAG lesson. Scanned/image-only PDFs are out of scope.

**Purpose of Code:** This function converts the uploaded handbook from a PDF into plain text. The assistant needs this text so it can search the handbook and use its policies when answering student questions.


In [117]:
def read_handbook_pdf(pdf_path: str) -> str:
    """Return readable text extracted from all pages of the handbook PDF."""
    # TODO 1: Open the PDF using PdfReader.
    page_text = []
    doc = PdfReader(pdf_path)
    # TODO 2: Extract and collect text from each page.

    for page in doc.pages:
      text = page.extract_text()
      if text and text.strip():
        page_text.append(text)

    # TODO 3: Combine the page text and return it.
    text = ' '.join(page_text)
    return text
    pass

handbook_text = read_handbook_pdf(PDF_PATH)
print("Extracted characters:", len(handbook_text))
print("First 100 characters:", handbook_text[:100])
print("Last 100 characters:", handbook_text[-100:])

Extracted characters: 3086
First 100 characters: FICTIONAL TRAINING DOCUMENT | GenAI Foundations Assignment
Page 1
Northstar Learning Hub
Student Han
Last 100 characters: questions must be grounded in
the provided text. If a topic is not covered, do not invent a policy.



### Step 3. Split the document into chunks

**Your task:** Complete `split_text_into_chunks(text, chunk_size, chunk_overlap)`.

- Split the text into smaller, overlapping character-based chunks.
- Start from character index `0` and move by `chunk_size - chunk_overlap`.
- Remove empty chunks and return a list of strings.
- Check that overlap is smaller than chunk size.

**Expected output:** A list such as `['first chunk ...', 'second chunk ...']`.

**Hint:** The RAG lesson used a simple `start`, `end`, and `while` loop. Do not use a new text-splitting library.

**Purpose of Code:** This function breaks the long handbook text into smaller, overlapping sections. Smaller chunks make it easier to find relevant policy information without sending the entire handbook to the LLM.


In [118]:
def split_text_into_chunks(
    text: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
) -> list[str]:
    """Return overlapping character-based chunks from the PDF text."""
    # TODO 1: Validate that 0 <= chunk_overlap < chunk_size.
    if not 0 <= chunk_overlap < chunk_size:
      raise ValueError("chunk_overlap must be between 0 and chunk_size")
    # TODO 2: Slice the text using a moving start and end position.
    start = 0
    end = chunk_size
    chunks = []
    while start < len(text):
      chunks.append(text[start:end])
      start += chunk_size - chunk_overlap
      end += chunk_size - chunk_overlap
    # TODO 3: Collect non-empty chunks, then return them.
    chunks = [chunk for chunk in chunks if chunk.strip()]
    return chunks

handbook_chunks = split_text_into_chunks(handbook_text)
print("Number of chunks:", len(handbook_chunks))
print("First chunk:", handbook_chunks[0])
print("Last chunk:", handbook_chunks[-1])
print(handbook_chunks[1])

Number of chunks: 4
First chunk: FICTIONAL TRAINING DOCUMENT | GenAI Foundations Assignment
Page 1
Northstar Learning Hub
Student Handbook | Version 1.0 | Fictional policy document for classroom assignments
1. Program Overview
Northstar Learning Hub provides an eight-week, instructor-led online training program. All dates and
deadlines follow India Standard Time (IST). Learners should read announcements in the learning portal
every week.
2. Enrollment and Orientation
New learners receive their portal login and an orientation email within two business days after
enrollment confirmation. Orientation includes course navigation and how to contact the support team.
3. Class Attendance
Live sessions take place on weekdays according to the cohort timetable. To qualify for a completion
certificate, a learner must attend at least 80% of scheduled live sessions. Recordings are available for
revision but do not count as live attendance.
4. Weekly Assignment Deadlines
Assignments for each teaching

## Part B — Apply Prompt Engineering
### Step 4. Create a zero-shot question-classification prompt

**Your task:** Complete `build_zero_shot_prompt(question)`.

Write a prompt that asks the LLM to classify a question using **only** these categories:
`Enrollment`, `Assignments`, `Certificates`, `Support`, `Other`.

Requirements:
- Give a clear instruction but **no examples**.
- Include the student's question.
- Ask for **only the category name** in the response.

**Example input:** `"Can I resubmit my graded homework?"`  
**Expected category:** `Assignments`.

**Purpose of Code:** This function creates a zero-shot prompt that tells the LLM how to categorize a student question without examples. It provides a baseline for comparing different prompt-engineering techniques.


In [121]:
def build_zero_shot_prompt(question: str) -> str:
    """Return a zero-shot classification prompt for a student question."""
    # TODO 1: Explain the classification task and allowed labels.

    # TODO 2: Insert the student's question.
    # TODO 3: Specify the exact output format.
    return f"""
    your job is to classify the user's question.
    pick exactly one of these categories: Enrollment, Assignments, Certificates, Support, Other.
    user question : {question}
    reply with only category name in the response
    """

print(build_zero_shot_prompt("Can I resubmit my graded homework?"))



    your job is to classify the user's question.
    pick exactly one of these categories: Enrollment, Assignments, Certificates, Support, Other.
    user question : Can I resubmit my graded homework?
    reply with only category name in the response
    


### Step 5. Create a few-shot question-classification prompt

**Your task:** Complete `build_few_shot_prompt(question)`.

Use the **same categories** and output format as Step 4, but add at least **three labeled examples** in the prompt.

You may use these examples:

| Example question | Category |
|---|---|
| How do I enroll in the next batch? | Enrollment |
| When is my assignment due? | Assignments |
| I cannot open my learning portal. | Support |

End with the new question and ask for its category. Return the complete prompt as a string.

**Purpose of Code:** This function creates a few-shot prompt with labeled examples of student questions. You will compare it with the zero-shot version to observe how examples can influence classification.


In [122]:
def build_few_shot_prompt(question: str) -> str:
    """Return a few-shot classification prompt with labeled examples."""
    # TODO 1: State the allowed categories.
    # TODO 2: Include three or more question/category examples.
    # TODO 3: Add the new question and request only the category.
    return f"""
    your job is to classify the user's question.
    pick exactly one of these categories: Enrollment, Assignments, Certificates, Support, Other.

    Example question: How do I enroll in the next batch?
    Category: Enrollment

    Example question: When is my assignment due?
    Category: Assignments

    Example question: I cannot open my learning portal.
    Category: Support

    user question : {question}
    reply with only category name in the response
    """
print(build_few_shot_prompt("Can I resubmit my graded homework?"))


    your job is to classify the user's question.
    pick exactly one of these categories: Enrollment, Assignments, Certificates, Support, Other.

    Example question: How do I enroll in the next batch?
    Category: Enrollment

    Example question: When is my assignment due?
    Category: Assignments

    Example question: I cannot open my learning portal.
    Category: Support

    user question : Can I resubmit my graded homework?
    reply with only category name in the response
    


## Part C — Call an LLM Using an API
### Step 6. Implement a reusable DeepSeek API function

**Your task:** Complete `call_deepseek(messages, temperature, max_tokens)` using **Python `requests`**, as in the LLM API lesson.

- Prepare the JSON request body with `model`, `messages`, `temperature`, and `max_tokens`.
- Send the request with `requests.post` to `DEEPSEEK_API_URL` using `HEADERS` and `timeout=60`.
- Call `raise_for_status()`.
- Parse the response JSON and return the assistant's message content as a string.
- Include any model-specific settings only if needed by your configured DeepSeek model.

**Expected input:** A list of `system` and `user` messages.  
**Expected output:** Generated text.

**Security:** Keep the API key in Colab Secrets; do not print it.

**Purpose of Code:** This function sends messages to the DeepSeek model and retrieves its response. Making it reusable allows both the classification tasks and the RAG pipeline to call the same API without duplicating request code.


In [123]:
def call_nvidia(
    messages: list[dict],
    temperature: float = 0.2,
    max_tokens: int = 250,
) -> str:
    """Send messages to DeepSeek and return generated assistant text."""
    # TODO 1: Build the JSON payload.
    payload = {
        "model": NVIDIA_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    # TODO 2: Send an HTTP POST with requests.
    response = requests.post(NVIDIA_API_URL, headers=HEADERS, json=payload, timeout=60)
    response.raise_for_status()
    # TODO 3: Check status, decode JSON, return the response text.
    response = response.json()
    return response["choices"][0]["message"]["content"]

print(call_nvidia([{"role": "user", "content": "Say hello in one word."}]))

Hello


### Step 7. Compare zero-shot and few-shot results

**Your task:** Complete `compare_classification_prompts(question)`.

- Call your zero-shot prompt function.
- Call your few-shot prompt function.
- Send each prompt to your `call_deepseek` function using the same low temperature.
- Return a dictionary containing `question`, `zero_shot`, and `few_shot`.

**Discussion:** Were the two results identical? If not, why might examples change the result?

**Note:** This step sends two API requests and may incur a small cost.

**Purpose of Code:** This function runs the same student question through zero-shot and few-shot prompts. Its output helps you compare their classifications and understand how prompt design affects LLM behavior.


In [124]:
def compare_classification_prompts(question: str) -> dict:
    """Return outputs from zero-shot and few-shot classification."""
    # TODO 1: Build both prompts.
    zero_shot_prompt = build_zero_shot_prompt(question)
    few_shot_prompt = build_few_shot_prompt(question)
    # TODO 2: Call DeepSeek separately for each prompt.
    zero_shot = call_nvidia([{"role": "user", "content": zero_shot_prompt}])
    few_shot = call_nvidia([{"role": "user", "content": few_shot_prompt}])
    # TODO 3: Return {'question': ..., 'zero_shot': ..., 'few_shot': ...}.
    return {
        'question': question,
        'zero_shot': zero_shot,
        'few_shot': few_shot
    }

print(compare_classification_prompts("Can I submit my homework late?"))

{'question': 'Can I submit my homework late?', 'zero_shot': 'Assignments', 'few_shot': 'Assignments'}


## Part D — Retrieve Relevant Context
### Step 8. Find the most relevant PDF chunk

**Your task:** Complete `retrieve_best_chunk(question, chunks)`.

Use **basic keyword overlap**, as demonstrated in the RAG lesson:
1. Convert the question to lowercase words using `re.findall(r"[a-zA-Z]+", ...)`.
2. Convert each chunk to lowercase words using the same method.
3. Score each chunk by the number of **shared unique words**.
4. Return the best chunk and its integer score as `(context, score)`.
5. If there are no chunks or every chunk has score `0`, return `("", 0)`.

**Do not use:** embeddings, vector databases, semantic-search packages, or the model to retrieve text.

**Why return a score?** It lets the caller notice obvious retrieval failures. A positive overlap score does **not** guarantee that the chunk contains the answer.

**Purpose of Code:** This function searches the handbook chunks for words that overlap with the student’s question. It selects useful context for the LLM without requiring embeddings or a vector database, and reports when no words match.


In [125]:
def retrieve_best_chunk(question: str, chunks: list[str]) -> tuple[str, int]:
    """Return the highest-overlap chunk and its keyword-overlap score."""
    # TODO 1: Convert the question to a set of lowercase words.
    question_words = set(re.findall(r"[a-zA-Z]+", question.lower()))
    # TODO 2: Compare it with words from every chunk.
    best_chunk = ""
    best_score = 0
    for chunk in chunks:
      chunk_words = set(re.findall(r"[a-zA-Z]+", chunk.lower()))
      score = len(question_words & chunk_words)
      if score > best_score:
        best_chunk = chunk
        best_score = score
    # TODO 3: Return the chunk with highest score, or ('', 0) if none match.
    return (best_chunk, best_score)

print(retrieve_best_chunk("What is the deadline for each weekly assignment?", handbook_chunks))
print(retrieve_best_chunk("Can I get my money back?", handbook_chunks))

('ilable for\nrevision but do not count as live attendance.\n4. Weekly Assignment Deadlines\nAssignments for each teaching week are due by Sunday at 11:59 PM IST. A learner may request an\nextension of up to 24 hours by email before the deadline. The instructor must approve the extension in\nwriting. Submissions without approval after the deadline are marked late.\n5. Resubmissions\nA learner may resubmit each graded assignment once, within seven calendar days after receiving\nfeedback. The resubmission replaces the original for assessment. Additional resubmissions are not\npermitted under this handbook.\n6. Completion Certificate\nTo receive a course completion certificate, learners must satisfy all three conditions: at least 80% live\nattendance, submission of every mandatory assignment, and a score of at least 60% on the final\nassessment. Meeting only one or two conditions is insufficient.\n FICTIONAL TRAINING DOCUMENT | GenAI Foundations Assignment\nPage 2\n7. Mentoring Support\nE

## Part E — Create the RAG Chain Using LangChain
### Step 9. Build the grounded prompt template

**Your task:** Complete `create_rag_prompt()` using `PromptTemplate`.

Your template must contain the variables `{context}` and `{question}` and tell the model:
- Answer **only** from the supplied handbook context.
- If the answer is not supported, respond **exactly**: `Insufficient context.`
- Give a short, student-friendly response.
- Do not invent policy details or assume any missing information.

**Expected output:** A LangChain `PromptTemplate` object.

**Purpose of Code:** This function defines the instructions and placeholders for the final RAG question-answering prompt. It tells the LLM to use the retrieved handbook context and avoid inventing answers when the context is insufficient.


In [126]:
#from tempfile import template
def create_rag_prompt() -> PromptTemplate:
    """Create and return a grounded RAG prompt template."""
    # TODO 1: Write a template with {context} and {question}.

    # TODO 2: Specify how to handle missing information.
    # TODO 3: Return a PromptTemplate object.
    template = """
    Answer only from the supplied handbook context.
    if the answer is not supported, respond exactly: Insufficient context.
    Give a short, student-friendly response.
    Do not invent policy details or assume any missing information.
    context: {context}
    question: {question}
    answer:
    """
    return PromptTemplate.from_template(template)

rag_prompt = create_rag_prompt()
print(rag_prompt.format(context="Assignments are due Sunday 11:59 PM IST", question="When is the deadline"))


    Answer only from the supplied handbook context.
    if the answer is not supported, respond exactly: Insufficient context.
    Give a short, student-friendly response.
    Do not invent policy details or assume any missing information.
    context: Assignments are due Sunday 11:59 PM IST
    question: When is the deadline
    answer:
    


### Step 10. Convert a LangChain prompt into a DeepSeek API response

**Your task:** Complete `deepseek_model_step(prompt_value)`.

- LangChain passes a prompt value to your model step.
- Convert it into text using `.to_string()`.
- Pass that text into `call_deepseek` as a user message, together with a concise system message.
- Return the generated text.

**Hint:** In the next step you will wrap this Python function using `RunnableLambda`.

**Expected output:** A string. Do not create a new API key or a second client.

**Purpose of Code:** This function acts as the link between LangChain and the DeepSeek API. It converts a LangChain-formatted prompt into an API request and returns the model’s answer to the chain.


In [127]:
def nvidia_model_step(prompt_value: Any) -> str:
    """Convert a LangChain prompt value into a nvidia text response."""
    # TODO 1: Convert prompt_value to a plain string.

    prompt = prompt_value.to_string()

    # TODO 2: Build system and user messages.
    system_message = "You are a helpful assistant."

    # TODO 3: Call call_nvidia and return the response.
    response = call_nvidia([{"role": "system", "content": system_message}, {"role": "user", "content": prompt}])
    return response


print(nvidia_model_step(rag_prompt.invoke({"context": "...", "question": "..."})))

Please provide the handbook context and the question you would like me to answer.


### Step 11. Combine the LangChain components

**Your task:** Complete `build_rag_chain(prompt)`.

Connect three components in this order:

```text
PromptTemplate -> RunnableLambda(deepseek_model_step) -> StrOutputParser
```

- Wrap `deepseek_model_step` in a `RunnableLambda`.
- Create a `StrOutputParser`.
- Use the LangChain pipe operator (`|`) to connect the components.
- Return the assembled chain.

**Expected output:** An object that supports `.invoke({'context': '...', 'question': '...'})`.

**Purpose of Code:** This function connects prompt preparation, the DeepSeek call, and output parsing into one reusable LangChain pipeline. The completed chain can answer different questions by receiving new context and question values.


In [128]:
def build_rag_chain(prompt: PromptTemplate):
    """Return PromptTemplate -> model runnable -> text parser chain."""
    # TODO 1: Create RunnableLambda(nvidia_model_step).
    model_step = RunnableLambda(nvidia_model_step)
    # TODO 2: Create StrOutputParser().
    output_parser = StrOutputParser()
    # TODO 3: Connect all three components with | and return the chain.
    chain = (prompt | model_step | output_parser)
    return chain
    pass

chain = build_rag_chain(rag_prompt)
print(chain.invoke({"context": "Assignments are due Sunday 11:59 PM IST.", "question": "When is the deadline?"}))


Assignments are due Sunday at 11:59 PM IST.


## Part F — Connect and Test the Assistant
### Step 12. Answer one student question

**Your task:** Complete `answer_student_question(question, chunks, chain)`.

- Call `retrieve_best_chunk`.
- If the best score is `0`, return `Insufficient context.` **without** an LLM request.
- Otherwise call `chain.invoke` with `{'context': context, 'question': question}`.
- Return this dictionary:

```python
{
    'question': '...',
    'answer': '...',
    'retrieval_score': 0,
    'retrieved_context': '...'
}
```

**Important:** Even with a positive retrieval score, the model must say `Insufficient context.` when the retrieved chunk does not actually answer the question. Do not treat keyword matches as proof.

**Purpose of Code:** This function combines retrieval and answer generation into the student helpdesk workflow. It searches the handbook, skips the LLM call when no chunk matches, and returns an answer with information about the retrieved context.


In [129]:
def answer_student_question(question: str, chunks: list[str], chain) -> dict:
    """Retrieve handbook context, run the chain, and return a result dict."""
    # TODO 1: Retrieve the best context and overlap score.
    context, score = retrieve_best_chunk(question, chunks)
    # TODO 2: Return insufficient context if retrieval failed.
    if score == 0:
      return {
          'question': question,
          'answer': 'Insufficient context.',
          'retrieval_score': score,
          'retrieved_context': context
      }
    # TODO 3: Otherwise invoke the chain using context and question.
    result = chain.invoke({'context': context, 'question': question})
    # TODO 4: Return the required result dictionary.
    return {
        'question': question,
        'answer': result,
        'retrieval_score': score,
        'retrieved_context': context
    }
    pass

print(answer_student_question("What is the deadline for each weekly assignment?", handbook_chunks, chain))

#testing with 2 questions

for que in ["What is the minimum live attendance required for a certificate?", "Does the academy provide students with free laptops?"]:
  r = answer_student_question(que, handbook_chunks, chain)
  print(r["answer"], "|" , r["retrieval_score"])

{'question': 'What is the deadline for each weekly assignment?', 'answer': 'Assignments for each teaching week are due by **Sunday at 11:59 PM IST**. You can request a 24-hour extension by email before the deadline, but it must be approved in writing by the instructor.', 'retrieval_score': 7, 'retrieved_context': 'ilable for\nrevision but do not count as live attendance.\n4. Weekly Assignment Deadlines\nAssignments for each teaching week are due by Sunday at 11:59 PM IST. A learner may request an\nextension of up to 24 hours by email before the deadline. The instructor must approve the extension in\nwriting. Submissions without approval after the deadline are marked late.\n5. Resubmissions\nA learner may resubmit each graded assignment once, within seven calendar days after receiving\nfeedback. The resubmission replaces the original for assessment. Additional resubmissions are not\npermitted under this handbook.\n6. Completion Certificate\nTo receive a course completion certificate, le

### Step 13. Run a functional test after implementing your functions

**Your task:** Execute this provided test harness. Fix any failures in your functions before continuing.

It checks extraction and chunking, compares the two classification prompts, builds the chain, and asks two representative policy questions.

**Note:** Run only after completing Steps 2–12. The code below is intentionally **not** a solution to any TODO function.

**Purpose of Code:** This test cell checks whether the functions you completed work together as one assistant. It verifies basic PDF processing, compares classification prompts, and runs sample questions through the RAG chain.


In [130]:
# TEST HARNESS — Run once all TODO functions are complete.
handbook_text = read_handbook_pdf(PDF_PATH)
assert isinstance(handbook_text, str) and handbook_text.strip(), "PDF text must not be empty."

handbook_chunks = split_text_into_chunks(handbook_text)
assert isinstance(handbook_chunks, list) and handbook_chunks, "Chunks must be a non-empty list."

print("Extracted characters:", len(handbook_text))
print("Number of chunks:", len(handbook_chunks))

comparison = compare_classification_prompts("Can I submit my homework late?")
print("\nPrompt comparison:", comparison)

rag_prompt = create_rag_prompt()
rag_chain = build_rag_chain(rag_prompt)

for sample_question in [
    "What is the minimum live attendance required for a certificate?",
    "Does the academy provide students with free laptops?",
]:
    result = answer_student_question(sample_question, handbook_chunks, rag_chain)
    print("\nQuestion:", sample_question)
    print("Answer:", result["answer"])
    print("Retrieval score:", result["retrieval_score"])

Extracted characters: 3086
Number of chunks: 4

Prompt comparison: {'question': 'Can I submit my homework late?', 'zero_shot': 'Assignments', 'few_shot': 'Assignments'}

Question: What is the minimum live attendance required for a certificate?
Answer: You need at least **80% live attendance** to qualify for the completion certificate.
Retrieval score: 7

Question: Does the academy provide students with free laptops?
Answer: Insufficient context.
Retrieval score: 3


### Step 14. Evaluate the assistant using a fixed test set

Use the provided six questions. For each response:
- Verify its claims directly against the handbook PDF.
- Label it `Supported`, `Unsupported`, or `Insufficient context` based on the **content**, not just the model's confidence.
- Write a one-sentence observation.
- Highlight at least one limitation of the simple word-overlap retriever.

**Your task:** Complete `evaluate_questions(test_questions, chunks, chain)` to collect the question and result for every test case. You can print your observations or write a Markdown results table below.

**Purpose of Code:** This function runs the assistant on a fixed set of student questions and collects its results. You will use those results to check whether answers are supported by the handbook and identify retrieval limitations.


In [131]:
TEST_QUESTIONS = [
    "What is the deadline for each weekly assignment?",
    "Can a student resubmit a graded assignment?",
    "What attendance is needed for a completion certificate?",
    "How many mentoring sessions can a learner book each month?",
    "Does the program guarantee a job?",
    "Does the academy give every student a free laptop?",
]

def evaluate_questions(test_questions: list[str], chunks: list[str], chain) -> list[dict]:
    """Return a list of assistant result dictionaries for the test questions."""
    # TODO 1: Loop through the test questions.
    all_results = []
    for question in test_questions:
      # TODO 2: Call answer_student_question for each one.
      result = answer_student_question(question, chunks, chain)
      # TODO 3: Append each result to a list and return it.
      all_results.append(result)
    return all_results
    pass

results = evaluate_questions(TEST_QUESTIONS, handbook_chunks, rag_chain)
for result in results:
  print(result)

{'question': 'What is the deadline for each weekly assignment?', 'answer': 'Assignments for each teaching week are due by **Sunday at 11:59 PM IST**.', 'retrieval_score': 7, 'retrieved_context': 'ilable for\nrevision but do not count as live attendance.\n4. Weekly Assignment Deadlines\nAssignments for each teaching week are due by Sunday at 11:59 PM IST. A learner may request an\nextension of up to 24 hours by email before the deadline. The instructor must approve the extension in\nwriting. Submissions without approval after the deadline are marked late.\n5. Resubmissions\nA learner may resubmit each graded assignment once, within seven calendar days after receiving\nfeedback. The resubmission replaces the original for assessment. Additional resubmissions are not\npermitted under this handbook.\n6. Completion Certificate\nTo receive a course completion certificate, learners must satisfy all three conditions: at least 80% live\nattendance, submission of every mandatory assignment, and a

### Step 15. Record and explain your findings (Markdown answer)

**Purpose of this step:** This final step does not require new code. It documents what your completed assistant produced, checks its answers against the handbook, and explains the strengths and limitations you observed.

Fill in the table below after running your evaluation function. This final step tests your ability to **interpret results**, not just run code.

| Test question (short name) | Answer supported by handbook? | Your observation |
|---|---|---|
| Weekly deadline | supported | Matches in section 4: (due sunday at 11:59PM IST.) |
| Resubmission | supported | Matches in section 5: (once within seven calender days after receiving feedback) |
| Certificate attendance | supported | Matches section 6(80% attendance, all mandatory assignments, 60% on the final assessment); the 80% also appears in section3 (liveclasses). |
| Mentoring | supported | Matches in section 7: upto two individual sessions per month, booked at least 24 hours ahead through the portal. |
| Job guarantee | Supported | Matches section 10: no guarentee of employment, interview or job placement, only optional resume review and interview practice. |
| Free laptop | Insufficient context | Handbook never mentions free laptops,, the retrival still scored 4 from common words ("the","a", "does", "student"), but model replied correctly "Insufficient context." |

**Answer these questions:**
1. Did zero-shot and few-shot prompting produce the same category? Explain briefly.

A. yes, in step 7 both returned "Assignments", and they matched also the question , which was clearly about homework.

2. Why does the assistant retrieve a chunk before calling the LLM?

A. without retrival the model would have to read complete pdf for every question, which costs more, its slower and long handbook wouldn't fit. with retrival, retrieve_best_chunk looks up the most relevant chunk and only that chunk goes to the model, together with the question

3. Give one example of a question that keyword-overlap retrieval might handle poorly.

A. here if user asks "can i get my money back?" instead of refund, the words dont match so the score is low or zero, which can be seen in step 8.
another example where there is no claim for free laptop sothe assistant returned "insufficient context",even though model generated the output because of common words

4. Did your assistant invent any policy? If so, how did you improve the prompt or test?

A. No policy was invented in the six test questions: the five answers with handbook content matched sections 4, 5, 6, 7 and 10.
  For the laptop question, the retriver returned an irrelevant chunk, but model replied "insufficient context." because the step 9 prompt tells it to answer only from the context.

## Submission Checklist


Upload your completed `.ipynb` with executed outputs on GitHub and share the link of repository.



- [ ] All 12 unfinished Python functions have been implemented (Steps 2–12 and Step 14).
- [ ] The notebook runs sequentially from a fresh Colab runtime after adding the API key and uploading the PDF.
- [ ] The zero-shot/few-shot comparison has an output.
- [ ] The PDF has been extracted and chunked successfully.
- [ ] A LangChain pipeline produces a grounded answer.
- [ ] The assistant handles a question that is not answered in the PDF.
- [ ] All six evaluation questions and the reflection are completed.
- [ ] The DeepSeek API key is not embedded in code or visible in notebook output.

### Assessment Rubric

| Criterion | Marks |
|---|---:|
| Zero-shot and few-shot prompt quality | 20 |
| Direct LLM API integration | 20 |
| LangChain chain construction | 20 |
| PDF processing, retrieval, grounded answers | 25 |
| Testing, observations, and submission quality | 15 |
| **Total** | **100** |

### Helpful references
- DeepSeek API: https://api-docs.deepseek.com/
- LangChain: https://python.langchain.com/
- PyPDF: https://pypdf.readthedocs.io/
